In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import load_model
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau


In [2]:
IMG_SIZE = 128
BATCH_SIZE = 16
NUM_CLASSES = 7

In [4]:
#Load Dataset

train_df = pd.read_csv("Datasets/train.csv")
valid_df = pd.read_csv("Datasets/validation.csv")
test_df = pd.read_csv("Datasets/test.csv")

print(train_df.shape)
print(valid_df.shape)
print(test_df.shape)
metadata = pd.read_csv("Datasets/HAM10000_metadata.csv")

metadata.head()

(7010, 11)
(1502, 11)
(1503, 11)


,lesion_id,image_id,dx,dx_type,age,sex,localization
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear


In [5]:
image_dir1 = "Datasets/HAM10000_images_part_1"
image_dir2 = "Datasets/HAM10000_images_part_2"

image_path = {}

for folder in [image_dir1, image_dir2]:

    for file in os.listdir(folder):

        image_id = file.split(".")[0]

        image_path[image_id] = os.path.join(folder, file)

metadata["path"] = metadata["image_id"].map(image_path)

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0027419.jpg
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025030.jpg
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0026769.jpg
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025661.jpg
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2/ISIC_0031633.jpg


In [6]:
encoder = LabelEncoder()

metadata["label"] = encoder.fit_transform(metadata["dx"])

metadata.head()

,lesion_id,image_id,dx,dx_type,age,sex,localization,path,label
0,HAM_0000118,ISIC_0027419,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0027419.jpg,2
1,HAM_0000118,ISIC_0025030,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025030.jpg,2
2,HAM_0002730,ISIC_0026769,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0026769.jpg,2
3,HAM_0002730,ISIC_0025661,bkl,histo,80.0,male,scalp,Datasets/HAM10000_images_part_1/ISIC_0025661.jpg,2
4,HAM_0001466,ISIC_0031633,bkl,histo,75.0,male,ear,Datasets/HAM10000_images_part_2/ISIC_0031633.jpg,2


In [7]:
train_df, temp_df = train_test_split(

    metadata,

    test_size=0.30,

    stratify=metadata["label"],

    random_state=42
)

val_df, test_df = train_test_split(

    temp_df,

    test_size=0.50,

    stratify=temp_df["label"],

    random_state=42
)

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

(7010, 9)
(1502, 9)
(1503, 9)


In [8]:
train_df["label"] = train_df["label"].astype(str)
val_df["label"] = val_df["label"].astype(str)
test_df["label"] = test_df["label"].astype(str)

In [9]:
train_df["path"] = train_df["path"].astype(str)
val_df["path"] = val_df["path"].astype(str)
test_df["path"] = test_df["path"].astype(str)

In [10]:
classes = np.unique(train_df["label"])

weights = compute_class_weight(

    class_weight="balanced",

    classes=classes,

    y=train_df["label"]
)

class_weights = dict(zip(range(len(classes)), weights))

print(class_weights)

{0: np.float64(4.37305053025577), 1: np.float64(2.7817460317460316), 2: np.float64(1.3022478172023035), 3: np.float64(12.36331569664903), 4: np.float64(1.285530900421786), 5: np.float64(0.21338772031292808), 6: np.float64(10.115440115440116)}


In [11]:
train_datagen = ImageDataGenerator(

    rescale=1./255,

    rotation_range=20,

    width_shift_range=0.1,

    height_shift_range=0.1,

    zoom_range=0.2,

    horizontal_flip=True,

    vertical_flip=True
)

test_datagen = ImageDataGenerator(

    rescale=1./255
)

In [12]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [28]:
train_datagen = ImageDataGenerator(

    

    rotation_range=20,

    horizontal_flip=True,

    vertical_flip=True,

    zoom_range=0.2,

    brightness_range=[0.8,1.2]

)



# EarlyStop

In [36]:
from tensorflow.keras.applications.efficientnet import preprocess_input

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    horizontal_flip=True,
    vertical_flip=True,
    zoom_range=0.2,
    brightness_range=[0.8,1.2]
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

test_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

# NOW create the generators
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [37]:
#EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
 
# Base Model
base_model = EfficientNetB0(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(128,128,3)
 
)
 
base_model.trainable = False
 
# Build Model
efficientnet = Sequential([
 
    base_model,
 
    GlobalAveragePooling2D(),
 
    Dense(256, activation="relu"),
 
    Dropout(0.5),
 
    Dense(NUM_CLASSES, activation="softmax")
 
])
 
efficientnet.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_3      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [38]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True,

    verbose=1

)

In [41]:
from tensorflow.keras.optimizers import Adam

efficientnet.compile(

    optimizer=Adam(learning_rate=1e-4),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [42]:
history_es = efficientnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=15,

    callbacks=[early_stop],
    class_weight=class_weights
    
    

)

Epoch 1/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - accuracy: 0.3330 - loss: 1.7762 - val_accuracy: 0.5060 - val_loss: 1.4405
Epoch 2/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.4491 - loss: 1.5061 - val_accuracy: 0.5626 - val_loss: 1.2433
Epoch 3/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 88ms/step - accuracy: 0.4937 - loss: 1.3848 - val_accuracy: 0.5093 - val_loss: 1.3040
Epoch 4/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 88ms/step - accuracy: 0.5003 - loss: 1.2970 - val_accuracy: 0.6232 - val_loss: 1.0745
Epoch 5/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 88ms/step - accuracy: 0.5224 - loss: 1.2566 - val_accuracy: 0.5579 - val_loss: 1.1669
Epoch 6/15
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.5241 - loss: 1.2496 - val_accuracy: 0.5746 - val_loss: 1.1403
Epoch 6: early stopping
Restoring model weights from the end of the best epoch: 4.


In [43]:
train_pred = efficientnet.predict(test_generator)
train_pred_classes = np.argmax(train_pred, axis=1)
print(np.unique(train_pred_classes, return_counts=True))

94/94 ━━━━━━━━━━━━━━━━━━━━ 7s 66ms/step
(array([0, 1, 2, 3, 4, 5, 6]), array([ 65, 117, 163,  77, 227, 781,  73]))


In [44]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet.evaluate(test_generator, verbose=0)

pred = efficientnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [45]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.6018545031547546, 0.6231691241264343, 0.6240851879119873, 0.7262370561520193, 0.6240851630073186, 0.6611298418812442]


In [46]:
efficientnet.save("efficientnet_early.keras")

# lR

In [47]:
#EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
 
# Base Model
base_model = EfficientNetB0(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(128,128,3)
 
)
 
base_model.trainable = False
 
# Build Model
efficientnet = Sequential([
 
    base_model,
 
    GlobalAveragePooling2D(),
 
    Dense(256, activation="relu"),
 
    Dropout(0.5),
 
    Dense(NUM_CLASSES, activation="softmax")
 
])
 
efficientnet.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_4      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [48]:


early_stop = EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True,

    verbose=1

)

In [49]:
from tensorflow.keras.callbacks import ReduceLROnPlateau

lr_scheduler = ReduceLROnPlateau(

    monitor="val_loss",

    factor=0.5,

    patience=2,

    min_lr=1e-7,

    verbose=1

)

In [50]:
from tensorflow.keras.optimizers import Adam

efficientnet.compile(

    optimizer=Adam(learning_rate=0.0005),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [51]:
history_lr = efficientnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    callbacks=[lr_scheduler, early_stop],class_weight=class_weights

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 46s 91ms/step - accuracy: 0.3884 - loss: 1.6794 - val_accuracy: 0.5999 - val_loss: 1.1695 - learning_rate: 5.0000e-04
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.4893 - loss: 1.3840 - val_accuracy: 0.4740 - val_loss: 1.2899 - learning_rate: 5.0000e-04
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.5171 - loss: 1.2919 - val_accuracy: 0.6172 - val_loss: 1.0315 - learning_rate: 5.0000e-04
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 88ms/step - accuracy: 0.5086 - loss: 1.2246 - val_accuracy: 0.4973 - val_loss: 1.2031 - learning_rate: 5.0000e-04
Epoch 5/5
438/439 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.5392 - loss: 1.1614
Epoch 5: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.5391 - loss: 1.1612 - val_accuracy: 0.5446 - val_loss: 1.1405 - learning_rate: 5.0000e-04
Epoch 5: early stopping
Restoring model weights from the end of

In [52]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet.evaluate(test_generator, verbose=0)

pred = efficientnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [53]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.6059914231300354, 0.6171770691871643, 0.6014637351036072, 0.7318539562376051, 0.6014637391882901, 0.6462962111603311]


In [54]:
efficientnet.save("efficient_lr.keras")

# SGD

In [55]:
#EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
 
# Base Model
base_model = EfficientNetB0(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(128,128,3)
 
)
 
base_model.trainable = False
 
# Build Model
efficientnet = Sequential([
 
    base_model,
 
    GlobalAveragePooling2D(),
 
    Dense(256, activation="relu"),
 
    Dropout(0.5),
 
    Dense(NUM_CLASSES, activation="softmax")
 
])
 
efficientnet.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_5      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [56]:
from tensorflow.keras.optimizers import SGD

efficientnet.compile(

    optimizer=SGD(
        learning_rate=0.01,
        momentum=0.9
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [57]:
history_sgd = efficientnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,class_weight=class_weights

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 45s 90ms/step - accuracy: 0.2699 - loss: 2.0667 - val_accuracy: 0.2557 - val_loss: 1.8432
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.2672 - loss: 1.8339 - val_accuracy: 0.1784 - val_loss: 1.7724
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.3063 - loss: 1.7869 - val_accuracy: 0.5053 - val_loss: 1.3990
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.2922 - loss: 1.7596 - val_accuracy: 0.4028 - val_loss: 1.5059
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 38s 87ms/step - accuracy: 0.2496 - loss: 1.7767 - val_accuracy: 0.4334 - val_loss: 1.5451


In [58]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet.evaluate(test_generator, verbose=0)

pred = efficientnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [59]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.3921540677547455, 0.43342211842536926, 0.4457751214504242, 0.6438063392124487, 0.4457751164337991, 0.5022855458505106]


In [60]:
efficientnet.save("efficientnet_sgd.keras")

# RMSprop

In [61]:
#EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
 
# Base Model
base_model = EfficientNetB0(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(128,128,3)
 
)
 
base_model.trainable = False
 
# Build Model
efficientnet = Sequential([
 
    base_model,
 
    GlobalAveragePooling2D(),
 
    Dense(256, activation="relu"),
 
    Dropout(0.5),
 
    Dense(NUM_CLASSES, activation="softmax")
 
])
 
efficientnet.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_6      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [62]:
from tensorflow.keras.optimizers import RMSprop

efficientnet.compile(

    optimizer=RMSprop(
        learning_rate=0.001
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [63]:
history_eff  = efficientnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 48s 96ms/step - accuracy: 0.6899 - loss: 0.9380 - val_accuracy: 0.7284 - val_loss: 0.7735
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 40s 91ms/step - accuracy: 0.7148 - loss: 0.8412 - val_accuracy: 0.7317 - val_loss: 0.7784
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 88ms/step - accuracy: 0.7178 - loss: 0.8192 - val_accuracy: 0.7417 - val_loss: 0.7499
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.7308 - loss: 0.8148 - val_accuracy: 0.7423 - val_loss: 0.7789
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 39s 89ms/step - accuracy: 0.7321 - loss: 0.8094 - val_accuracy: 0.7523 - val_loss: 0.7461


In [64]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet.evaluate(test_generator, verbose=0)

pred = efficientnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [65]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7623395323753357, 0.7523302435874939, 0.7504990100860596, 0.727282194591617, 0.7504990019960079, 0.7241626040961582]


In [66]:
efficientnet.save("efficientnet_rms.keras")

In [111]:
efficient_hyper = load_model("efficientnet_rms.keras")

In [112]:
train_pred = efficient_hyper.predict(train_generator)
train_pred_classes = np.argmax(train_pred, axis=1)
print(np.unique(train_pred_classes, return_counts=True))

220/220 ━━━━━━━━━━━━━━━━━━━━ 36s 158ms/step
(array([0, 1, 2, 3, 4, 5, 6]), array([ 155,  328,  549,    4,  606, 5330,   38]))


# Batchwise Comparison

In [99]:
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 5

In [100]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


In [101]:
#EfficientNetB0
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam
 
# Base Model
base_model = EfficientNetB0(
 
    weights="imagenet",
 
    include_top=False,
 
    input_shape=(128,128,3)
 
)
 
base_model.trainable = False
 
# Build Model
efficientnet = Sequential([
 
    base_model,
 
    GlobalAveragePooling2D(),
 
    Dense(256, activation="relu"),
 
    Dropout(0.5),
 
    Dense(NUM_CLASSES, activation="softmax")
 
])
 
efficientnet.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 329,735 (1.26 MB)

 Non-trainable params: 4,049,571 (15.45 MB)

In [102]:
from tensorflow.keras.optimizers import RMSprop

efficientnet.compile(

    optimizer=RMSprop(
        learning_rate=0.001
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [103]:
history = efficientnet.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 54s 215ms/step - accuracy: 0.6849 - loss: 0.9315 - val_accuracy: 0.7310 - val_loss: 0.7453
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 48s 219ms/step - accuracy: 0.7225 - loss: 0.8047 - val_accuracy: 0.7330 - val_loss: 0.7563
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 47s 213ms/step - accuracy: 0.7241 - loss: 0.7804 - val_accuracy: 0.7310 - val_loss: 0.7471
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 43s 195ms/step - accuracy: 0.7324 - loss: 0.7568 - val_accuracy: 0.7563 - val_loss: 0.6928
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 39s 176ms/step - accuracy: 0.7409 - loss: 0.7429 - val_accuracy: 0.7510 - val_loss: 0.7097


In [104]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet.evaluate(test_generator, verbose=0)

pred = efficientnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [105]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.7761768698692322, 0.7509986758232117, 0.7378576397895813, 0.7004950923307213, 0.737857618097139, 0.6894609488528872]


In [106]:
efficientnet.save("efficientnet_batch.keras")

In [85]:
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5

In [86]:
train_generator = train_datagen.flow_from_dataframe(

    dataframe=train_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=True
)

val_generator = test_datagen.flow_from_dataframe(

    dataframe=val_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

test_generator = test_datagen.flow_from_dataframe(

    dataframe=test_df,

    x_col="path",

    y_col="label",

    target_size=(IMG_SIZE, IMG_SIZE),

    batch_size=BATCH_SIZE,

    class_mode="categorical",

    shuffle=False
)

Found 7010 validated image filenames belonging to 7 classes.
Found 1502 validated image filenames belonging to 7 classes.
Found 1503 validated image filenames belonging to 7 classes.


# finetuning

In [75]:
from tensorflow.keras.applications import EfficientNetB0

base_model = EfficientNetB0(

    weights="imagenet",

    include_top=False,

    input_shape=(IMG_SIZE, IMG_SIZE, 3)

)

In [76]:
base_model.trainable = True

for layer in base_model.layers[:-10]:

    layer.trainable = False

In [77]:
print("Trainable Layers:")

for layer in base_model.layers[-10:]:

    print(layer.name, layer.trainable)

Trainable Layers:
block7a_se_squeeze True
block7a_se_reshape True
block7a_se_reduce True
block7a_se_expand True
block7a_se_excite True
block7a_project_conv True
block7a_project_bn True
top_conv True
top_bn True
top_activation True


In [78]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import GlobalAveragePooling2D
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Dropout

inputs = Input(
    shape=(IMG_SIZE, IMG_SIZE, 3)
)

x = base_model(
    inputs,
    training=False
)

x = GlobalAveragePooling2D()(x)

x = Dense(
    256,
    activation="relu"
)(x)

x = Dropout(0.3)(x)

outputs = Dense(
    NUM_CLASSES,
    activation="softmax"
)(x)

efficientnet_ft = Model(
    inputs,
    outputs
)

efficientnet_ft.summary()

Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_17 (InputLayer)     │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 4, 4, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_8      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,379,306 (16.71 MB)

 Trainable params: 1,222,967 (4.67 MB)

 Non-trainable params: 3,156,339 (12.04 MB)

In [79]:
from tensorflow.keras.optimizers import Adam

efficientnet_ft.compile(

    optimizer=Adam(
        learning_rate=1e-5
    ),

    loss="categorical_crossentropy",

    metrics=["accuracy"]

)

In [80]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(

    monitor="val_loss",

    patience=2,

    restore_best_weights=True,

    verbose=1

)

In [81]:
history_ft = efficientnet_ft.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights,

    callbacks=[early_stop]

)

Epoch 1/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 50s 194ms/step - accuracy: 0.2466 - loss: 2.0401 - val_accuracy: 0.2723 - val_loss: 1.7921
Epoch 2/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 39s 178ms/step - accuracy: 0.2805 - loss: 1.8851 - val_accuracy: 0.3003 - val_loss: 1.7341
Epoch 3/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 39s 177ms/step - accuracy: 0.3087 - loss: 1.7851 - val_accuracy: 0.3269 - val_loss: 1.6961
Epoch 4/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 39s 176ms/step - accuracy: 0.3282 - loss: 1.7025 - val_accuracy: 0.3635 - val_loss: 1.6522
Epoch 5/5
220/220 ━━━━━━━━━━━━━━━━━━━━ 40s 182ms/step - accuracy: 0.3596 - loss: 1.6367 - val_accuracy: 0.3802 - val_loss: 1.6182
Restoring model weights from the end of the best epoch: 5.


In [84]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet_ft.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet_ft.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet_ft.evaluate(test_generator, verbose=0)

pred = efficientnet_ft.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

In [87]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.42952924966812134, 0.38015979528427124, 0.3666001260280609, 0.6536026822541324, 0.3666001330671989, 0.4336499884484398]


In [88]:
efficientnet.save("efficientnet_fine.keras")

# hyperparametr 


In [89]:
import keras_tuner as kt

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    GlobalAveragePooling2D,
    Dense,
    Dropout
)
from tensorflow.keras.optimizers import Adam, RMSprop

In [90]:
def build_efficientnet(hp):

    base_model = EfficientNetB0(

        weights="imagenet",

        include_top=False,

        input_shape=(IMG_SIZE, IMG_SIZE, 3)

    )

    base_model.trainable = False

    inputs = Input(
        shape=(IMG_SIZE, IMG_SIZE, 3)
    )

    x = base_model(
        inputs,
        training=False
    )

    x = GlobalAveragePooling2D()(x)

    x = Dense(

        units=hp.Choice(
            "dense_units",
            [128, 256]
        ),

        activation="relu"

    )(x)

    x = Dropout(

        hp.Choice(
            "dropout",
            [0.3, 0.5]
        )

    )(x)

    outputs = Dense(
        NUM_CLASSES,
        activation="softmax"
    )(x)

    model = Model(
        inputs,
        outputs
    )

    optimizer_name = hp.Choice(
        "optimizer",
        ["adam", "rmsprop"]
    )

    lr = hp.Choice(
        "learning_rate",
        [0.001, 0.0001]
    )

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)

    model.compile(

        optimizer=optimizer,

        loss="categorical_crossentropy",

        metrics=["accuracy"]

    )

    return model

In [91]:
tuner = kt.RandomSearch(

    build_efficientnet,

    objective="val_accuracy",

    max_trials=3,

    directory="efficientnet_tuning",

    project_name="efficientnet_hp"
)

In [92]:
tuner.search(

    train_generator,

    validation_data=val_generator,

    epochs=3

)

Trial 3 Complete [00h 02m 41s]
val_accuracy: 0.7190412878990173

Best val_accuracy So Far: 0.7223701477050781
Total elapsed time: 00h 07m 31s


In [93]:
best_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print(best_hps.values)

{'dense_units': 256, 'dropout': 0.5, 'optimizer': 'adam', 'learning_rate': 0.0001}


In [94]:
best_model = tuner.hypermodel.build(
    best_hps
)

history = best_model.fit(

    train_generator,

    validation_data=val_generator,

    epochs=5,

    class_weight=class_weights

)

Epoch 1/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 55s 110ms/step - accuracy: 0.3455 - loss: 1.8064 - val_accuracy: 0.4787 - val_loss: 1.4751
Epoch 2/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 50s 113ms/step - accuracy: 0.4561 - loss: 1.5251 - val_accuracy: 0.5905 - val_loss: 1.2270
Epoch 3/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 50s 113ms/step - accuracy: 0.4890 - loss: 1.4192 - val_accuracy: 0.5593 - val_loss: 1.2473
Epoch 4/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 49s 111ms/step - accuracy: 0.4976 - loss: 1.3008 - val_accuracy: 0.5945 - val_loss: 1.1447
Epoch 5/5
439/439 ━━━━━━━━━━━━━━━━━━━━ 50s 113ms/step - accuracy: 0.5110 - loss: 1.2988 - val_accuracy: 0.5905 - val_loss: 1.1502


In [96]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

train_loss, train_acc = efficientnet.evaluate(train_generator, verbose=0)

val_loss, val_acc =efficientnet.evaluate(val_generator, verbose=0)

test_loss, test_acc =efficientnet.evaluate(test_generator, verbose=0)

pred = efficientnet.predict(test_generator, verbose=0)

pred_labels = np.argmax(pred, axis=1)

true_labels = test_generator.classes

precision = precision_score(
    true_labels,
    pred_labels,
    average="weighted"
)

recall = recall_score(
    true_labels,
    pred_labels,
    average="weighted"
)

f1 = f1_score(
    true_labels,
    pred_labels,
    average="weighted"
)

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [97]:
print([
        train_acc,
        val_acc,
        test_acc,
        precision,
        recall,
        f1
    ])

[0.1306704729795456, 0.12849533557891846, 0.12641383707523346, 0.49255225477887554, 0.1264138389886893, 0.12128743174772832]


In [98]:
efficientnet.save("efficientnet_hype.keras")

In [109]:
comparison_df = pd.DataFrame({
    "Model": [
        "Early Stopping",
        "Learning Rate Scheduler",
        "SGD",
        "RMSprop",
        "Fine-Tuning",
        "Hyperparameter Tuning",
        "Batch-wise Comparison"
    ],
    "Train Accuracy": [
        0.6018545031547546,
        0.6059914231300354,
        0.3921540677547455,
        0.7623395323753357,
        0.42952924966812134,
        0.1306704729795456,
        0.7761768698692322
    ],
    "Validation Accuracy": [
        0.6231691241264343,
        0.6171770691871643,
        0.43342211842536926,
        0.7523302435874939,
        0.38015979528427124,
        0.12849533557891846,
        0.7509986758232117
    ],
    "Test Accuracy": [
        0.6240851879119873,
        0.6014637351036072,
        0.4457751214504242,
        0.7504990100860596,
        0.3666001260280609,
        0.12641383707523346,
        0.7378576397895813
    ],
    "Precision": [
        0.7262370561520193,
        0.7318539562376051,
        0.6438063392124487,
        0.727282194591617,
        0.6536026822541324,
        0.49255225477887554,
        0.7004950923307213
    ],
    "Recall": [
        0.6240851630073186,
        0.6014637391882901,
        0.4457751164337991,
        0.7504990019960079,
        0.3666001330671989,
        0.1264138389886893,
        0.737857618097139
    ],
    "F1-Score": [
        0.6611298418812442,
        0.6462962111603311,
        0.5022855458505106,
        0.7241626040961582,
        0.4336499884484398,
        0.12128743174772832,
        0.6894609488528872
    ]
})

comparison_df = (
    comparison_df.sort_values("Test Accuracy", ascending=False)
                 .reset_index(drop=True)
                 .round(4)
)

comparison_df

,Model,Train Accuracy,Validation Accuracy,Test Accuracy,Precision,Recall,F1-Score
0,RMSprop,0.7623,0.7523,0.7505,0.7273,0.7505,0.7242
1,Batch-wise Comparison,0.7762,0.7510,0.7379,0.7005,0.7379,0.6895
2,Early Stopping,0.6019,0.6232,0.6241,0.7262,0.6241,0.6611
3,Learning Rate Scheduler,0.6060,0.6172,0.6015,0.7319,0.6015,0.6463
4,SGD,0.3922,0.4334,0.4458,0.6438,0.4458,0.5023
5,Fine-Tuning,0.4295,0.3802,0.3666,0.6536,0.3666,0.4336
6,Hyperparameter Tuning,0.1307,0.1285,0.1264,0.4926,0.1264,0.1213
